In [1]:
import xgboost as xgb
print(xgb.__version__)

3.1.3


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

C:\Users\arramana\personal\projects\end-to-end-MlOps\.venv\Scripts\python.exe
3.1.3
C:\Users\arramana\personal\projects\end-to-end-MlOps\.venv\Lib\site-packages\xgboost\__init__.py


In [5]:
# 1. Imports
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [8]:
# 2. Load processed datasets
train_df = pd.read_csv("C:/Users/arramana/personal/projects/end-to-end-MlOps/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("C:/Users/arramana/personal/projects/end-to-end-MlOps/data/processed/feature_engineered_eval.csv")

In [9]:
# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [10]:
# 3. Define Optuna objective function with MLflow
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators",20, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train,y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse":rmse,"mae":mae,"r2":r2})
    
    return rmse

In [14]:
# 4. Run Optuna study with MLflow
# Force MLflow to always use the root project mlruns folder

mlflow.set_tracking_uri("file:///C:/Users/arramana/personal/projects/end-to-end-MlOps/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials=15) # 15 runs

print("Best params: ",study.best_trial.params)

[I 2026-02-22 15:22:54,103] A new study created in memory with name: no-name-3f68633d-7877-4b41-9b6f-7558c3f94fc4
[I 2026-02-22 15:23:40,867] Trial 0 finished with value: 81021.8362931256 and parameters: {'n_estimators': 340, 'max_depth': 9, 'learning_rate': 0.1531912821501054, 'subsample': 0.9547930267104043, 'colsample_bytree': 0.5999535852611333, 'min_child_weight': 7, 'gamma': 0.07725783495455607, 'reg_alpha': 0.001064603129292466, 'reg_lambda': 1.4745995104707106e-06}. Best is trial 0 with value: 81021.8362931256.
[I 2026-02-22 15:24:35,075] Trial 1 finished with value: 70050.44162996749 and parameters: {'n_estimators': 651, 'max_depth': 6, 'learning_rate': 0.03490646817810391, 'subsample': 0.6475930325350929, 'colsample_bytree': 0.6642951935519393, 'min_child_weight': 7, 'gamma': 1.801749754069521, 'reg_alpha': 8.458316216743128, 'reg_lambda': 4.833533492157586}. Best is trial 1 with value: 70050.44162996749.
[I 2026-02-22 15:26:16,199] Trial 2 finished with value: 71246.51516666

Best params:  {'n_estimators': 671, 'max_depth': 10, 'learning_rate': 0.028669718468836806, 'subsample': 0.5230348128914155, 'colsample_bytree': 0.511654199216631, 'min_child_weight': 5, 'gamma': 1.380406488838184, 'reg_alpha': 0.1416141470665974, 'reg_lambda': 6.742259995515929e-05}


In [15]:
# 5. Train final model with best params and log to MLflow

best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train,y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_pred,y_eval)
rmse = np.sqrt(mean_squared_error(y_pred,y_eval))
r2 = r2_score(y_pred,y_eval)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name = "best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse":rmse, "mae":mae, "r2":r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 30598.082176271793
RMSE: 70128.4994982748
R²: 0.9593276092099405


2026/02/22 15:44:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
